### DATASET TYPES 

Three main dataset types , each designed for different data processing 


1. STREAMING TABLES (ST

- A table with support for streaming or incremental data processing, only processing new data as it arrives , rather than reprocessing everything each time the pipeline runs. This approach significantly increase efficiency and decrease cost , especially when working with large or frequently updated data 

INCREMENTAL PROCESSING MODES 

- Supports both batch r Streaming for incremental processing (exactly one processing of files)

EFFiCIENT DATA UPDATES 

- Each time a streaming table is refreshed , data added to the source tables is appended to the streaming table

SQL COMMAND USEAGE 

- CREATE OR REFRESH STREAMING TABLE 

STREAMING READ SYNTAX
- Must use from STREAM read_files() to enable incremental streaming reads with Checkpointing 

AUTO LOADER INTEGRATION 

- This syntax leverage Datarbicks AutoLoader , which automatically tracks new files and ensure reliable, incremental ingestion 

NO DUPLICATE READS 

- File names are guaranteed to be read only once, with the goal of not having duplicate reads with in your incremental ingestion 




2. Materialized Views 

- Records are processed as required to return accurate results for the current data state
- Used for data processing tasks such as 
- Transformations 
- Aggregations
- Pre computing slow queries 
- Frequently used computations 

DYNAMIC QUERY RECALCULATION 

- Each time a materialized view is updated , query results are recalculated to reflect changes in Upstream datasets

PIPELINE DRIVEN MAINTENACNE 

- Automated created and updated by the pipeline 

SQL COMMAND USEAGE 

- Use the CREATE OR REFRESH MATERIALIZED VIEW syntax

FLEXIBALE PIPELINE PLACEMENT 

- Can be used anywhere in your pipeline , not just in a Gold layer

INCREMENTAL REFRESH CAPABILITY 

- Where applicable , results are incrementally refreshed , avoiding a full rebuild when new data arrives , Supported on Serverless Compute

COST BASED OPTIMIZATION 

- Incremental refresh is driven by a cost optimizer to power fast and efficient transformations on Serverless Compute 





3. Views 

- Construct a virtual table with no physical data based on the query in your declarative pipelines 
- Temporary view 
- View 

TEMPORARY AND PIPELINED SCOPED 

- Exists only during the pipeline run and is not registered in Unity Catalog 
- CREATE TEMPORARY VIEW 

VIRTUAL TABLE NO STORAGE 

- CREATE from sql query results without storing physical data 
- Stored in Unity catalog and created using CRETE VIEW 

### STREAMING TABLE reading from JSON Files 

In this example, we use the CREATE OR REFRESH STREAMING TABLE statement to create a streaming table called orders_bronze in the 1_bronze_db schema.

The SELECT clause defines which columns you want to pull into the streaming table from your source data.

In the FROM clause:

The STREAM keyword tells the pipeline to use streaming semantics, meaning it will process new files incrementally as they arrive.
The read_files() function points to the location of your source files. It reads the files and returns the contents in a tabular format.
This setup ensures that only new data is processed each time the pipeline runs, enabling efficient, scalable ingestion directly from cloud storage.

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE 1_bronze_db.orders_bronze AS
SELECT
  *,
  current_timestamp() AS processing_time,
  _metadata.file_name AS source_file
FROM STREAM read_files(
  "{{ source_path }}/orders",
  format => 'JSON');

### Streaming table orders_silver reading from orders_bronze

The SELECT clause is where you apply your SQL transformation logic to the streaming data. In this example, we’re keeping it simple, but in practice these transformations can include complex joins, filters, column derivations, and more.
The most important part is in the FROM clause. Here, you use the STREAM keyword with the name of the source streaming table, in this case, orders_bronze. This tells the pipeline to incrementally transform only the new rows that have landed in the bronze table since the last run.

This setup helps ensure that your silver table always stays up to date with the latest processed data, without reprocessing anything that’s already been handled.

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE 2_silver_db.orders_silver AS
SELECT
  order_id,
  timestamp(order_timestamp) AS order_timestamp,
  customer_id,
  notifications
FROM STREAM 1_bronze_db.orders_bronze;

### Materialized view gold_orders_by_date summarizing orders_silver
We use the CREATE OR REFRESH MATERIALIZED VIEW statement to define this view inside the 3_gold_db schema.

In the FROM clause, we reference the source table, orders_silver, without the STREAM keyword. That’s because Materialized Views automatically track changes and manage refreshes based on the upstream streaming table.

Important note: Where possible, the system will use incremental refreshes to update the view efficiently, rather than rebuilding it from scratch. This is supported in Serverless compute and driven by a cost-based optimizer for performance.

This approach is ideal for creating gold-layer aggregations or business-ready outputs with minimal overhead and high performance.

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW 3_gold_db.gold_orders_by_date AS
SELECT
  date(order_timestamp) AS order_date,
  count(*) AS total_daily_orders
FROM 2_silver_db.orders_silver
GROUP BY date(order_timestamp);

### DECLARATIVE PIPLEINE 
The orders_bronze table ingests the raw JSON data.
The orders_silver streaming table is linked to orders_bronze, transforming the ingested data incrementally.
The gold_orders_by_date materialized view depends on orders_silver streaming table, aggregating the transformed data for downstream business use.
This automatic linking simplifies pipeline development and reduces errors by removing the need to manually manage execution order. It also makes it easy to view and monitor your pipeline and its dependencies, giving you clear visibility into how data flows through each stage.

![Screenshot 2026-09-05 at 9.27.36 AM.png](./Screenshot 2026-09-05 at 9.27.36 AM.png "Screenshot 2026-09-05 at 9.27.36 AM.png")

### ENSURE DATA QUALITY WITH EXPECTATIONS 

Explain what expectations are and how they are used to enforce data quality rules in Apache Spark™ Declarative Pipelines
Write the CONSTRAINT syntax to define data quality expectations on streaming tables
Describe the three violation actions — WARN, DROP, and FAIL — and explain when to use each
Apply expectations to a streaming table using SQL with all three violation actions
Trace how a row of data is evaluated through expectations in the pipeline and explain what happens when a constraint is violated

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE 2_silver_db.orders_silver
 (
   CONSTRAINT valid_notifications EXPECT (notifications IN ('Y','N')),
   CONSTRAINT valid_date EXPECT (order_timestamp > "2021-01-01") ON VIOLATION FAIL UPDATE,
   CONSTRAINT valid_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW
 )
AS
SELECT
  order_id,
  timestamp(order_timestamp) AS order_timestamp,
  customer_id,
  notifications
FROM STREAM 1_bronze_db.orders_bronze;

![Screenshot 2026-09-05 at 10.25.13 AM.png](./Screenshot 2026-09-05 at 10.25.13 AM.png "Screenshot 2026-09-05 at 10.25.13 AM.png")